# Experiment 10: Intra-Model Ensemble — E9 bias=0 + bias=+2 softmax average

**Zero additional training.** Loads E9's saved weights, runs inference at two bias levels, averages softmax outputs.

## Rationale

E9 demonstrated that a single converged model can produce multiple operating points via post-hoc logit bias. Head A (bias=0.0) delivers the best overall accuracy (93.26%); Head B (bias=+2.0) delivers the best Shirt TPR (0.834). Averaging their softmax outputs should retain most of Head A's accuracy while improving Shirt TPR — the two heads disagree on Shirt classification, so the ensemble softens the Shirt decision boundary (analogous to E7's label smoothing but without retraining).

In [1]:
import sys; sys.path.append('..')
import os, torch, torch.nn as nn
import numpy as np
from src.data_utils import get_dataloaders
from src.eval_utils import (
    evaluate_detailed, get_all_probas_and_labels,
    compute_roc_auc_scores, compute_pr_auc_scores
)
import torchvision.transforms as transforms

OUT_DIR = '../outputs/error_analysis/ensemble'
os.makedirs(OUT_DIR, exist_ok=True)

if torch.backends.mps.is_available():    device = 'mps'
elif torch.cuda.is_available():          device = 'cuda'
else:                                    device = 'cpu'
print(f'Device: {device}')

Device: cuda


## Dataset — identical to E1

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
test_ds = __import__('torchvision').datasets.FashionMNIST(root='../data', train=False, download=True, transform=transform)
class_names = test_ds.classes
SHIRT_IDX = 6

test_loader, _ = get_dataloaders(test_ds, test_ds, batch_size=64)
print(f'Test batches: {len(test_loader)}')

Test batches: 157


## Load E9 model weights

In [3]:
class DiagnosticCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1); self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, 3, padding=1); self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1); self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, 3, padding=1); self.bn4 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2)
        self.conv5 = nn.Conv2d(64, 128, 3, padding=1); self.bn5 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(128, num_classes)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.relu(self.bn3(self.conv3(x)))
        x = self.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.relu(self.bn5(self.conv5(x)))
        x = self.pool3(x)
        x = self.gap(x).view(x.size(0), -1)
        x = self.drop(x)
        return self.fc(x)

model = DiagnosticCNN().to(device)
model.load_state_dict(torch.load('../outputs/error_analysis/logit_adjustment/model_weights.pth', map_location=device))
model.eval()
print(f'Loaded E9 weights ({sum(p.numel() for p in model.parameters()):,} params)')

Loaded E9 weights (140,778 params)


c:\document\Study documents\Deeplearning_Course\.venv\Lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


## Ensemble inference — average softmax from two bias heads

In [4]:
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

BIAS_A, BIAS_B = 0.0, 2.0

all_preds_ensemble = []
all_labels = []
sh_tp_e = sh_fp_e = sh_fn_e = 0

with torch.no_grad():
    for inputs, lbls in test_loader:
        inputs, lbls = inputs.to(device), lbls.to(device)
        logits = model(inputs)
        logits_a = logits.clone(); logits_a[:, SHIRT_IDX] += BIAS_A
        logits_b = logits.clone(); logits_b[:, SHIRT_IDX] += BIAS_B
        probs_a = torch.softmax(logits_a, dim=1)
        probs_b = torch.softmax(logits_b, dim=1)
        probs_avg = (probs_a + probs_b) / 2.0
        preds = probs_avg.argmax(dim=1)

        all_preds_ensemble.extend(preds.cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())

        for true, pred in zip(lbls.cpu().numpy(), preds.cpu().numpy()):
            if pred == SHIRT_IDX and true == SHIRT_IDX: sh_tp_e += 1
            if pred == SHIRT_IDX and true != SHIRT_IDX: sh_fp_e += 1
            if pred != SHIRT_IDX and true == SHIRT_IDX: sh_fn_e += 1

acc_e = accuracy_score(all_labels, all_preds_ensemble)
tpr_e = sh_tp_e / (sh_tp_e + sh_fn_e + 1e-8)
prec_e = sh_tp_e / (sh_tp_e + sh_fp_e + 1e-8)
cm_e = confusion_matrix(all_labels, all_preds_ensemble)

print(f'Ensemble (bias=0 + bias=+2):')
print(f'  Accuracy:  {acc_e*100:.2f}%')
print(f'  Shirt TPR:  {tpr_e:.4f}')
print(f'  Shirt Prec: {prec_e:.4f}')

per_class_tpr_e = {}
for i, name in enumerate(class_names):
    tp = cm_e[i, i]; fn = cm_e[i].sum() - tp
    per_class_tpr_e[name] = tp / (tp + fn + 1e-8)
    print(f'  {name:<15} TPR={per_class_tpr_e[name]:.4f}')

Ensemble (bias=0 + bias=+2):
  Accuracy:  93.17%
  Shirt TPR:  0.8110
  Shirt Prec: 0.7776
  T-shirt/top     TPR=0.8680
  Trouser         TPR=0.9880
  Pullover        TPR=0.8930
  Dress           TPR=0.9360
  Coat            TPR=0.8950
  Sandal          TPR=0.9870
  Shirt           TPR=0.8110
  Sneaker         TPR=0.9770
  Bag             TPR=0.9890
  Ankle boot      TPR=0.9730


## Comparison: Ensemble vs individual heads vs best prior

In [5]:
# E9 individual heads from bias_sweep_results.txt
e9_data = {
    'Head A (bias=0)':  {'acc': 93.26, 'tpr': 0.7850, 'prec': 0.8060},
    'Head B (bias=+2)': {'acc': 93.00, 'tpr': 0.8340, 'prec': 0.7507},
}
prior = {
    'E1 (CE base)':   {'acc': 92.50, 'tpr': 0.847, 'prec': 0.723},
    'E7 (smooth)':    {'acc': 92.04, 'tpr': 0.879, 'prec': 0.692},
    'E8 (extended)':  {'acc': 92.99, 'tpr': 0.797, 'prec': 0.792},
}

header = f'{"Model":<22} {"Acc%":>7} {"ShirtTPR":>9} {"ShirtPrec":>10}'
print(header); print('-' * len(header))
print(f'{"E10 ensemble":<22} {acc_e*100:>7.2f} {tpr_e:>9.4f} {prec_e:>10.4f}')
for name, d in e9_data.items():
    print(f'{name:<22} {d["acc"]:>7.2f} {d["tpr"]:>9.4f} {d["prec"]:>10.4f}')
print('-' * len(header))
for name, d in prior.items():
    print(f'{name:<22} {d["acc"]:>7.2f} {d["tpr"]:>9.4f} {d["prec"]:>10.4f}')

Model                     Acc%  ShirtTPR  ShirtPrec
---------------------------------------------------
E10 ensemble             93.17    0.8110     0.7776
Head A (bias=0)          93.26    0.7850     0.8060
Head B (bias=+2)         93.00    0.8340     0.7507
---------------------------------------------------
E1 (CE base)             92.50    0.8470     0.7230
E7 (smooth)              92.04    0.8790     0.6920
E8 (extended)            92.99    0.7970     0.7920


## Save results

In [6]:
cm_np = cm_e

with open(os.path.join(OUT_DIR, 'metrics_summary.txt'), 'w') as f:
    f.write(f'Test Accuracy (percentage): {acc_e*100:.2f}\n')
    f.write(f'Test Accuracy (fraction): {acc_e:.4f}\n\n')
    f.write(f'{"Class":<15} {"TPR":>8} {"Precision":>10}\n')
    f.write('-' * 33 + '\n')
    for i, name in enumerate(class_names):
        tp = cm_np[i, i]; total = cm_np[i].sum()
        fp = cm_np[:, i].sum() - tp
        tpr = tp / (total + 1e-8)
        prec = tp / (tp + fp + 1e-8)
        f.write(f'{name:<15} {tpr:>8.4f} {prec:>10.4f}\n')

with open(os.path.join(OUT_DIR, 'confusion_matrix.txt'), 'w') as f:
    f.write(f'{"":>15}')
    for name in class_names:
        f.write(f'{name:>15}')
    f.write('\n')
    for i in range(10):
        f.write(f'{class_names[i]:>15}')
        for j in range(10):
            f.write(f'{cm_np[i, j]:>15}')
        f.write('\n')

with open(os.path.join(OUT_DIR, 'ensemble_comparison.txt'), 'w') as f:
    f.write(header + '\n' + '-' * len(header) + '\n')
    f.write(f'{"E10 ensemble":<22} {acc_e*100:>7.2f} {tpr_e:>9.4f} {prec_e:>10.4f}\n')
    for name, d in {**e9_data, **prior}.items():
        f.write(f'{name:<22} {d["acc"]:>7.2f} {d["tpr"]:>9.4f} {d["prec"]:>10.4f}\n')

print(f'\nAll results saved to {OUT_DIR}/')


All results saved to ../outputs/error_analysis/ensemble/
